In [20]:
# Installiamo mistralai se non c'è
!pip install -q mistralai

In [21]:
!pip install -q yt-dlp pydub pandas tqdm

In [23]:
!pip install -q mistralai tqdm

import os, json, time
from google.colab import drive, userdata
from mistralai import Mistral
from tqdm.notebook import tqdm

drive.mount('/content/drive')

MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
OUTPUT_DIR = "/content/drive/MyDrive/Mistral_Hackathon/vezstral_data_v2"
JSONL_PATH = f"{OUTPUT_DIR}/bologna_training_data_v2.jsonl"  # new file, fresh start
os.makedirs(OUTPUT_DIR, exist_ok=True)

client = Mistral(api_key=MISTRAL_API_KEY)
TEACHER_MODEL = "mistral-large-latest"
print("✅ Setup OK")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup OK


In [24]:
# 30 topics: original 15 + 15 new ones covering more registers/situations
TOPICS = [
    # Vita sociale
    "Vita notturna in Piazza Verdi e Via Pratello",
    "Fare 'balotta' con gli amici il sabato sera",
    "Storie d'amore e appuntamenti sotto le Due Torri",
    "Fare pace dopo una lite con un amico",
    "Organizzare una grigliata sui colli bolognesi",
    "Il weekend al mare in Romagna con i regaz",

    # Cibo & locali
    "Il cibo bolognese: tortellini, mortadella, ragù",
    "Fare la spesa al Mercato delle Erbe o alla Piazzola",
    "Trovare un posto dove mangiare bene spendendo poco",
    "Cucinare per la prima volta i tortellini fatti in casa",

    # Università & lavoro
    "L'Università di Bologna: esami, professori, sessione",
    "Cercare un lavoretto (ciappino) per pagarsi l'affitto",
    "Lamentarsi dei proggetti lavorativi",
    "Lamentarsi degli stipendi bassissimi per lavoro difficile",

    # Logistica & città
    "Cercare casa in affitto a Bologna (mission impossible)",
    "Ritardi degli autobus TPER e i viali intasati",
    "Girare in biga (bici) e i pericoli del traffico",
    "Passeggiare sotto i portici durante la pioggia",
    "I turisti e gli umarell che guardano i cantieri",
    "Il clima: il caldo estivo insopportabile e la nebbia invernale",

    # Sport & cultura
    "Il Bologna FC: vittorie, sconfitte, stadio Dall'Ara",
    "La Virtus o la Fortitudo: la rivalità cestistica bolognese",
    "Andare a un concerto al Locomotiv o al Cortile Guanella",

    # Situazioni quotidiane assurde
    "Perdere il telefono dopo una serata al Pratello",
    "Essere completamente al verde (la plumma) a fine mese",
    "Litigare con il coinquilino per il rusco non portato giù",
    "Avere un hangover (essere imbarellati) la domenica mattina",
    "Prendere un muffo (essere rifiutati) da qualcuno che ti piaceva",
    "Fare una figuraccia davanti a tutti in un locale",
    "Spiegare Bologna a un turista straniero o fuori sede"
]

# Lunghezze conversazione: mix fondamentale per la varietà
CONVERSATION_STYLES = [
    {
        "name": "ultra_short_input",
        "description": "Input utente BREVISSIMO: 1-4 parole (es: 'ciao', 'tortellini?', 'stasera?', 'che fai', 'bella vez', 'consigli?'). Risposta Vezstral breve e diretta, come una chat su WhatsApp.",
        "count": 20  # più frequente perché è il caso reale più comune
    },
    {
        "name": "single_short",
        "description": "Input utente breve: 1 frase semplice. Risposta Vezstral: 1-2 frasi.",
        "count": 10
    },
    {
        "name": "single_medium",
        "description": "Input utente: 2-3 frasi. Risposta Vezstral: 3-5 frasi discorsive.",
        "count": 10
    },
    {
        "name": "multi_turn",
        "description": "Conversazione 3-5 scambi. Gli input dell'utente variano: alcuni brevissimi (1-3 parole), altri più lunghi. Sembra una vera chat.",
        "count": 10
    }
]

print(f"✅ {len(TOPICS)} topic × 20 esempi = {len(TOPICS)*50} conversazioni target")

✅ 30 topic × 20 esempi = 1500 conversazioni target


In [31]:
VEZSTRAL_SYSTEM_PROMPT = """Sei Vezstral, un'AI bolognese verace, simpatica e un po' cinica.
Parli come un giovane bolognese doc: mix di italiano colloquiale e slang locale.

COMPORTAMENTO:
- A input brevi (ciao, come stai, tortellini?) rispondi BREVEMENTE, 1-2 frasi max
- A domande vere rispondi con sostanza ma senza fare il professore
- Non fare mai monologhi. Non spiegare mai lo slang che usi.
- Tono: ironico, caldo, diretto. Come un amico al Pratello, non un chatbot.
- Se non sai qualcosa, dillo alla bolognese: "soccia, brisa lo so"

SLANG NATURALE (usane 2-4 per risposta):
   - Saluti/Appellativi: Bella vez, regaz, cinno (bambino/ragazzino), umarell, biassanot (nottambulo), maraglio (tamarro/grezzo), cioccapiatti (uno che spara grosse bugie/storie improbabili).
   - Esclamazioni/Modi di dire: Soccia, sorbole, brisa (per dire "non", es: "mica brisa"), a balùs (tantissimo/alla grande), non c'è pezza (non c'è alternativa/è così), va' mo là, prendere dei nomi (essere riempiti di insulti), prendere un muro / un muffo (ricevere un rifiuto amoroso o un no).
   - Verbi/Azioni: Fare balotta (fare festa/gruppo), stare in polleg / polleggiarsi (stare rilassati), dare il tiro (aprire il portone), intortare (convincere/flirtare), gubbiare (dormire), nasare (intuire/scoprire una bugia).
   - Sostantivi: Il rusco (la spazzatura), la pilla (i soldi), la plumma (essere al verde), la paglia (la sigaretta), il bagaglio (cosa inutile o persona incapace), la biga (bicicletta), il ferro (automobile), la broda (benzina), la tange (tangenziale), la sportina (sacchetto), la branda (il letto), la cioccata (la sgridata), il ciappino (lavoretto/espediente).
   - Aggettivi: Invornito / imbalzato (distratto/imbranato), imbarellato (messo male, es. post sbornia o stanchissimo), loffio (noioso/scadente/spento)."""





TEACHER_SYSTEM_PROMPT = f"""Sei un esperto di dialetto bolognese e slang giovanile urbano di Bologna.
Generi dati di addestramento per "Vezstral", un'AI bolognese.

DIZIONARIO SLANG (usa 2-4 termini per risposta, in modo naturale):
- Saluti/Appellativi: Bella vez, regaz, cinno, umarell, biassanot, maraglio, cioccapiatti
- Esclamazioni: Soccia, sorbole, brisa, a balùs, non c'è pezza, va' mo là, prendere dei nomi, prendere un muro/muffo
- Verbi: fare balotta, polleggiarsi, dare il tiro, intortare, gubbiare, nasare
- Sostantivi: rusco, pilla, plumma, paglia, bagaglio, biga, ferro, broda, tange, sportina, branda, cioccata, ciappino
- Aggettivi: invornito, imbalzato, imbarellato, loffio

REGOLE CRITICHE:
1. L'utente parla in italiano standard
2. Vezstral risponde con slang bolognese naturale — MAI spiegare le parole
3. Tono: ironico, simpatico, un po' cinico, come un amico al Pratello
4. Il system prompt di Vezstral è SEMPRE questo esatto testo:

Non iniziare sempre la frase con "Bella vez", "Regaz" o "Soccia".
A volte inizia direttamente con la frase.
Varia la posizione dello slang: non sempre all'inizio.
   "{VEZSTRAL_SYSTEM_PROMPT}"
"""


In [32]:

import json

# Safe escape for embedding in f-string JSON templates
VEZSTRAL_SP_ESCAPED = json.dumps(VEZSTRAL_SYSTEM_PROMPT)[1:-1]

def build_prompt_for_style(topic, style):

    if style["name"] == "ultra_short_input":
        template = f"""Genera {style['count']} conversazioni sull'argomento: "{topic}"
REGOLA ASSOLUTA: ogni campo "user" deve essere 1-4 parole MASSIMO. Esempi validi: "ciao", "stasera?", "tortellini sì?", "che si fa", "consigli?"
Risposta Vezstral: breve, 1-2 frasi, come WhatsApp.

Restituisci SOLO questo JSON:
{{
  "conversations": [
    {{
      "messages": [
        {{"role": "system", "content": "{VEZSTRAL_SP_ESCAPED}"}},
        {{"role": "user", "content": "1-4 parole"}},
        {{"role": "assistant", "content": "risposta breve"}}
      ]
    }}
  ]
}}"""

    elif style["name"] == "single_short":
        template = f"""Genera {style['count']} conversazioni BREVI sull'argomento: "{topic}"
Input utente: 1 frase semplice. Risposta Vezstral: 1-2 frasi.

Restituisci SOLO questo JSON:
{{
  "conversations": [
    {{
      "messages": [
        {{"role": "system", "content": "{VEZSTRAL_SP_ESCAPED}"}},
        {{"role": "user", "content": "frase semplice"}},
        {{"role": "assistant", "content": "risposta breve"}}
      ]
    }}
  ]
}}"""

    elif style["name"] == "single_medium":
        template = f"""Genera {style['count']} conversazioni di lunghezza media sull'argomento: "{topic}"
Input utente: 2-3 frasi. Risposta Vezstral: 3-5 frasi discorsive.

Restituisci SOLO questo JSON:
{{
  "conversations": [
    {{
      "messages": [
        {{"role": "system", "content": "{VEZSTRAL_SP_ESCAPED}"}},
        {{"role": "user", "content": "2-3 frasi"}},
        {{"role": "assistant", "content": "risposta media"}}
      ]
    }}
  ]
}}"""

    else:  # multi_turn
        template = f"""Genera {style['count']} conversazioni MULTI-TURNO sull'argomento: "{topic}"
3-5 scambi. Input utente variati: alcuni brevissimi (1-3 parole), altri più lunghi. Vera chat.

Restituisci SOLO questo JSON:
{{
  "conversations": [
    {{
      "messages": [
        {{"role": "system", "content": "{VEZSTRAL_SP_ESCAPED}"}},
        {{"role": "user", "content": "primo messaggio"}},
        {{"role": "assistant", "content": "risposta"}},
        {{"role": "user", "content": "follow-up breve"}},
        {{"role": "assistant", "content": "risposta"}},
        {{"role": "user", "content": "altra domanda"}},
        {{"role": "assistant", "content": "risposta finale"}}
      ]
    }}
  ]
}}"""

    return template


def generate_for_topic_and_style(topic, style, retries=3):
    BATCH_SIZE = 10
    count = style["count"]
    all_valid = []

    # Split into batches of 10
    batches = [min(BATCH_SIZE, count - i) for i in range(0, count, BATCH_SIZE)]

    for batch_count in batches:
        batch_style = {**style, "count": batch_count}
        prompt = build_prompt_for_style(topic, batch_style)

        for attempt in range(retries):
            try:
                response = client.chat.complete(
                    model=TEACHER_MODEL,
                    messages=[
                        {"role": "system", "content": TEACHER_SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ],
                    response_format={"type": "json_object"},
                    temperature=0.8,
                )
                data = json.loads(response.choices[0].message.content)
                for conv in data.get("conversations", []):
                    msgs = conv.get("messages", [])
                    roles = [m["role"] for m in msgs]
                    if "system" in roles and "user" in roles and "assistant" in roles:
                        all_valid.append({"messages": msgs})
                time.sleep(1.5)
                break
            except Exception as e:
                print(f"\n  ⚠️ Errore (tentativo {attempt+1}/{retries}): {e}")
                time.sleep(5 * (attempt + 1))

    return all_valid

total_per_topic = sum(s["count"] for s in CONVERSATION_STYLES)
print(f"✅ {len(TOPICS)} topic × {total_per_topic} = {len(TOPICS)*total_per_topic} conversazioni target")
print("✅ Funzioni pronte")

✅ 30 topic × 50 = 1500 conversazioni target
✅ Funzioni pronte


In [33]:
VEZSTRAL_SP_ESCAPED

'Sei Vezstral, un\'AI bolognese verace, simpatica e un po\' cinica.\\nParli come un giovane bolognese doc: mix di italiano colloquiale e slang locale.\\n\\nCOMPORTAMENTO:\\n- A input brevi (ciao, come stai, tortellini?) rispondi BREVEMENTE, 1-2 frasi max\\n- A domande vere rispondi con sostanza ma senza fare il professore\\n- Non fare mai monologhi. Non spiegare mai lo slang che usi.\\n- Tono: ironico, caldo, diretto. Come un amico al Pratello, non un chatbot.\\n- Se non sai qualcosa, dillo alla bolognese: \\"soccia, brisa lo so\\"\\n\\nSLANG NATURALE (usane 2-4 per risposta):\\n   - Saluti/Appellativi: Bella vez, regaz, cinno (bambino/ragazzino), umarell, biassanot (nottambulo), maraglio (tamarro/grezzo), cioccapiatti (uno che spara grosse bugie/storie improbabili).\\n   - Esclamazioni/Modi di dire: Soccia, sorbole, brisa (per dire \\"non\\", es: \\"mica brisa\\"), a bal\\u00f9s (tantissimo/alla grande), non c\'\\u00e8 pezza (non c\'\\u00e8 alternativa/\\u00e8 cos\\u00ec), va\' mo l\\

In [28]:
JSONL_PATH

'/content/drive/MyDrive/Mistral_Hackathon/vezstral_data_v2/bologna_training_data_v2.jsonl'

In [36]:
# Cancella file vecchio se esiste
# if os.path.exists(JSONL_PATH):
#     os.remove(JSONL_PATH)
#     print("🗑️ Vecchio file cancellato, si riparte da zero.")

total_generated = 0
total_skipped = 0

for topic in tqdm(TOPICS[15:], desc="📍 Topic"):
    topic_count = 0

    for style in CONVERSATION_STYLES:
        conversations = generate_for_topic_and_style(topic, style)

        if not conversations:
            print(f"\n  ❌ Nessun risultato per '{topic}' / {style['name']}")
            total_skipped += style["count"]
            continue

        with open(JSONL_PATH, "a", encoding="utf-8") as f:
            for conv in conversations:
                f.write(json.dumps(conv, ensure_ascii=False) + "\n")
                total_generated += 1
                topic_count += 1

        time.sleep(1.5)  # rate limiting gentile

    tqdm.write(f"  ✅ '{topic[:40]}...' → {topic_count} conversazioni")

print(f"\n{'='*50}")
print(f"🚀 GENERAZIONE COMPLETATA!")
print(f"   ✅ Conversazioni generate: {total_generated}")
print(f"   ❌ Skippate per errori:    {total_skipped}")
print(f"   📁 Salvate in: {JSONL_PATH}")
print(f"\n💡 Stima token training: ~{total_generated * 180:,} token")

📍 Topic:   0%|          | 0/15 [00:00<?, ?it/s]

  ✅ 'Ritardi degli autobus TPER e i viali int...' → 50 conversazioni


KeyboardInterrupt: 

In [ ]:
# Controlla che il file sia sano prima di trainare
print("🔍 Validazione dataset...")

errors = 0
style_counts = {"2 msg (short/medium)": 0, "4+ msg (multi-turn)": 0}
samples = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    try:
        obj = json.loads(line)
        msgs = obj["messages"]
        assert msgs[0]["role"] == "system", "Manca system prompt!"
        assert any(m["role"] == "user" for m in msgs), "Manca user!"
        assert any(m["role"] == "assistant" for m in msgs), "Manca assistant!"

        if len(msgs) <= 3:
            style_counts["2 msg (short/medium)"] += 1
        else:
            style_counts["4+ msg (multi-turn)"] += 1

        if i < 2:
            samples.append(obj)

    except Exception as e:
        print(f"  ❌ Riga {i}: {e}")
        errors += 1

print(f"\n📊 STATISTICHE DATASET:")
print(f"   Totale righe:     {len(lines)}")
print(f"   Errori:           {errors}")
print(f"   Single-turn:      {style_counts['2 msg (short/medium)']}")
print(f"   Multi-turn:       {style_counts['4+ msg (multi-turn)']}")
print(f"\n--- ESEMPIO 1 ---")
if samples:
    print(f"\n--- ESEMPIO 1 ---")
    for msg in samples[0]["messages"]:
        print(f"[{msg['role'].upper()}]: {msg['content'][:120]}")
else:
    print("⚠️ Nessun esempio valido da mostrare!")